# FER 六分类 · v3 完整重训版

目标：在约两小时预算内争取超过你已提交的 **0.72**，以 **0.75** 为目标；**不是已实测达到的分数**。
本 Notebook 可在全新会话从头运行，不需要任何旧变量、旧权重或追加补丁。

**操作：** 导入这个完整文件，添加比赛数据，开启 GPU 和 Internet，从头运行全部单元。
使用单卡 cuda:0，不使用 DataParallel。先训练较快的 ResNet34，再训练 EfficientNetV2-S。
不要把这份完整 Notebook 粘在旧追加补丁下面混跑。

**结束后必须下载两个文件：**
1. `submission_v3.csv`：用于提交，必须为 7065 行、`ID,Emotion` 两列。
2. `fer6_v3_backup.zip`：含最佳/最后权重、配置、划分、训练日志、预测概率及提交，供恢复。

自动生成 ZIP 不等于永久保存：请下载到电脑；仅留在运行磁盘，后台会话结束后仍可能丢失。
原 `submission_v2.csv` 不会被覆盖，保留已经取得的 0.72 对照。
固定映射：0 愤怒、1 恐惧、2 快乐、3 悲伤、4 惊讶、5 中性。每天最多提交5次，组名用本人学号。


## 1. 配置
预计约 60–100 分钟，依据之前单卡速度估计，尚未实测本版 GPU 耗时。
总软预算 118 分钟，训练截止 99 分钟，为完整预测、报告和备份留 19 分钟。
按批次检查训练时间，单次验证、权重下载和磁盘写入仍可能使时间超出估计；不能保证任意硬件严格两小时。
默认 batch=64；显存不足时改为32；数据加载进程异常时把 NUM_WORKERS 改为0。
首次无需填写 RESUME_FROM。恢复方法见最后一节。


In [ ]:
import time
RUN_STARTED = time.monotonic()
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext
import copy
import gc
import hashlib
import json
import math
import os
import random
import shutil
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms as T
from torchvision.transforms import InterpolationMode

DATA_ROOT = Path('/kaggle/input/competitions/2026task2/fer_data/fer_data')
TRAIN_DIR, TEST_DIR = DATA_ROOT / 'train', DATA_ROOT / 'test'
SAMPLE_SUBMISSION = None
MANUAL_SUBMISSION_COLUMNS = ('ID', 'Emotion')
EXPECTED_TEST_ROWS = 7065
RUN_NAME = 'fer6_score_v3'
OUT = Path('/kaggle/working') / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
TOTAL_MINUTES = 118
PREDICTION_RESERVE_MINUTES = 19
TRAIN_DEADLINE = RUN_STARTED + (TOTAL_MINUTES - PREDICTION_RESERVE_MINUTES) * 60
BATCH_PER_GPU = BATCH_SIZE = 64   # 显存不足改成 32
NUM_WORKERS = 2                  # worker 异常改成 0
USE_MULTI_GPU = False
GPU_IDS = [0]
MULTISCALE_TTA = True
RESUME_FROM = None               # 首次不用填；恢复时可填备份 ZIP 或解压后的 fer6_score_v3 目录
OLD_RUN_DIR = None
LOCAL_PRETRAINED = {'resnet34': None, 'efficientnet_v2_s': None}
CLASS_NAMES = ['Angry', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
CLASS_ZH = ['愤怒', '恐惧', '快乐', '悲伤', '惊讶', '中性']
MODEL_CONFIGS = [
    dict(name='resnet34', epochs=32, size=160, alt_size=192, lr=2e-4,
         head_lr=6e-4, minutes=35.0, mixup_p=0.60, mixup_alpha=0.30,
         weight_decay=0.01, label_smoothing=0.03, dropout=0.4,
         warmup_epochs=1, bn_freeze_after=6, min_epochs=12, patience=7),
    dict(name='efficientnet_v2_s', epochs=32, size=192, alt_size=160, lr=2e-4,
         head_lr=8e-4, minutes=62.0, mixup_p=0.60, mixup_alpha=0.30,
         weight_decay=0.01, label_smoothing=0.03, dropout=0.4,
         warmup_epochs=1, bn_freeze_after=6, min_epochs=12, patience=7),
]

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('请在 Kaggle 启用 GPU，再从头运行本 Notebook。')
DEVICE = torch.device('cuda:0')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_num_threads(2)
print('V3 完整版 | 单卡：', torch.cuda.get_device_name(0), '| batch：', BATCH_SIZE)
print('总预算 118 分钟，训练最多 99 分钟，预测与备份预留 19 分钟。')
print('输出：', OUT, '| 不会覆盖 submission_v2.csv')


## 2. 自动恢复与备份工具
首次运行不需要旧文件。有明确的 v3 备份时才恢复，绝不把比赛模板当成模型输出。


In [ ]:
import zipfile
import stat

BACKUP_PATH = OUT.parent / 'fer6_v3_backup.zip'

def restore_backup(source):
    source = Path(source)
    if (OUT / 'run_config.json').is_file():
        print('当前输出已有实验记录，优先复用；不覆盖为外部备份。')
        return
    if any(OUT.iterdir()):
        raise RuntimeError('当前输出目录非空但缺少配置，请检查后使用新的 RUN_NAME。')
    if source.is_file() and source.suffix.lower() == '.zip':
        with zipfile.ZipFile(source) as archive:
            entries = archive.infolist()
            names = [entry.filename for entry in entries]
            if len(set(names)) != len(names):
                raise ValueError('备份ZIP存在重复条目。')
            if sum(entry.file_size for entry in entries) > 8 * 1024**3:
                raise ValueError('备份超出预期大小，请核对所选文件。')
            for entry in entries:
                path = Path(entry.filename)
                if (path.is_absolute() or '..' in path.parts or not path.parts
                        or path.parts[0] != RUN_NAME
                        or stat.S_ISLNK(entry.external_attr >> 16)):
                    raise ValueError('备份ZIP包含不符合预期的路径。')
            config_name = RUN_NAME + '/run_config.json'
            if config_name not in names:
                raise ValueError('该ZIP没有v3实验配置，不能恢复。')
            meta = json.loads(archive.read(config_name))
            if meta.get('version') != 3 or meta.get('models') != MODEL_CONFIGS:
                raise ValueError('备份版本/模型配置与当前Notebook不同。')
            archive.extractall(OUT.parent)
    elif source.is_dir():
        if source.resolve() == OUT.resolve():
            return
        meta = json.loads((source / 'run_config.json').read_text())
        if meta.get('version') != 3 or meta.get('models') != MODEL_CONFIGS:
            raise ValueError('备份版本/模型配置与当前Notebook不同。')
        shutil.copytree(source, OUT, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(f'没有找到恢复来源：{source}')
    print('已恢复文件；下一步仍会核对数据哈希和验证划分：', source)

def make_backup(stage, verify=False):
    if not (OUT / 'run_config.json').is_file():
        raise FileNotFoundError('缺少 run_config.json，不能创建可恢复备份。')
    files = sorted(p for p in OUT.rglob('*') if p.is_file() and not p.name.endswith('.tmp'))
    if not any(p.name == 'best.pt' for p in files):
        raise RuntimeError('尚无最佳模型，不能把空目录作为训练备份。')
    note = dict(stage=stage, models_with_best=[p.parent.name for p in files if p.name == 'best.pt'],
                elapsed_minutes=(time.monotonic() - RUN_STARTED) / 60,
                note='下载ZIP到电脑才完成独立备份；只保存在会话磁盘仍可能丢失。')
    (OUT / 'backup_status.json').write_text(json.dumps(note, ensure_ascii=False, indent=2))
    files = sorted(p for p in OUT.rglob('*') if p.is_file() and not p.name.endswith('.tmp'))
    temporary = BACKUP_PATH.with_suffix('.zip.tmp')
    print('正在打包权重与结果：', stage, flush=True)
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for path in files:
            archive.write(path, path.relative_to(OUT.parent).as_posix())
    if verify:
        with zipfile.ZipFile(temporary) as archive:
            damaged = archive.testzip()
            if damaged:
                raise IOError(f'备份校验失败：{damaged}')
            names = set(archive.namelist())
            required = {RUN_NAME + '/run_config.json', RUN_NAME + '/split.csv'}
            for config in MODEL_CONFIGS:
                folder = OUT / config['name']
                if (folder / 'best.pt').is_file():
                    required.update(RUN_NAME + '/' + config['name'] + '/' + item
                                    for item in ['best.pt', 'last.pt', 'history.csv'])
            if not required.issubset(names):
                raise IOError('备份缺少恢复所需文件。')
    os.replace(temporary, BACKUP_PATH)
    print('备份已更新：', BACKUP_PATH, '|', round(BACKUP_PATH.stat().st_size / 1024**2), 'MiB', flush=True)
    return BACKUP_PATH

if RESUME_FROM is None and not (OUT / 'run_config.json').is_file():
    candidates = sorted(Path('/kaggle/input').rglob('fer6_v3_backup.zip'))
    candidates += sorted(p.parent for p in Path('/kaggle/input').rglob('run_config.json')
                         if p.parent.name == RUN_NAME)
    if len(candidates) == 1:
        RESUME_FROM = str(candidates[0])
    elif len(candidates) > 1:
        raise RuntimeError('发现多个v3备份，请在 RESUME_FROM 中明确选择一个，避免误恢复。')
if RESUME_FROM:
    restore_backup(RESUME_FROM)
else:
    print('首次独立训练，或复用当前目录的v3记录；无需旧会话变量。')


## 3. 数据检查
按文件夹名称映射六类标签，检查损坏图片并缓存48×48灰度图。训练前强制检查测试图像共7065张。
优先读取匹配的提交模板；没有模板时使用完整文件名生成 `ID,Emotion`。不补占位标签，不丢弃测试图片。


In [ ]:
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
ALIASES = {
    'angry': 0, 'anger': 0, '愤怒': 0, '0': 0,
    'fear': 1, 'fearful': 1, '恐惧': 1, '1': 1,
    'happy': 2, 'happiness': 2, '快乐': 2, '高兴': 2, '2': 2,
    'sad': 3, 'sadness': 3, '悲伤': 3, '3': 3,
    'surprise': 4, 'surprised': 4, '惊讶': 4, '4': 4,
    'neutral': 5, '中性': 5, '5': 5,
}

def normalize_folder(name):
    return name.strip().casefold().replace(' ', '').replace('_', '').replace('-', '')

def resolve_data_dirs(train_dir, test_dir):
    if train_dir.is_dir() and test_dir.is_dir():
        return train_dir, test_dir
    # 兼容 Kaggle 的另一种常见挂载路径，但只接受唯一匹配项。
    candidates = [p for p in Path('/kaggle/input').rglob('train')
                  if p.is_dir() and (p.parent / 'test').is_dir()
                  and ('2026task2' in str(p) or 'fer_data' in str(p))]
    if len(candidates) != 1:
        raise FileNotFoundError(f'请检查比赛数据挂载路径。候选 train 目录：{candidates}')
    return candidates[0], candidates[0].parent / 'test'

def discover_images(train_dir, test_dir):
    paths, labels, seen_folders = [], [], {}
    for p in sorted(train_dir.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        relative = p.relative_to(train_dir)
        if len(relative.parts) < 2:
            raise ValueError(f'训练图片必须位于标签文件夹内：{p}')
        folder = relative.parts[0]
        key = normalize_folder(folder)
        if key not in ALIASES:
            raise ValueError(f'未知类别文件夹 {folder!r}。请核对六类映射，不能自动按字母排序。')
        seen_folders[folder] = ALIASES[key]
        paths.append(p)
        labels.append(ALIASES[key])
    test_paths = sorted(p for p in test_dir.rglob('*')
                        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    labels = np.asarray(labels, dtype=np.int64)
    if set(labels.tolist()) != set(range(6)) or not test_paths:
        raise ValueError('训练集必须含全部六类，测试集不能为空。')
    if len({p.name for p in test_paths}) != len(test_paths):
        raise ValueError('测试图片的文件名不唯一，请核对数据。')
    if len(test_paths) != EXPECTED_TEST_ROWS:
        raise ValueError(
            f'测试目录只发现 {len(test_paths)} 张图片，比赛需要 {EXPECTED_TEST_ROWS} 张。'
            f'请检查 TEST_DIR={test_dir}；此时不要训练，也不要用占位标签补齐。')
    return paths, labels, test_paths, seen_folders

def align_sample_ids(ids, test_paths, test_dir):
    lookup = {}
    for i, p in enumerate(test_paths):
        rel = p.relative_to(test_dir).as_posix()
        for key in {rel, p.name, p.stem, 'test/' + rel,
                    Path(rel).with_suffix('').as_posix()}:
            if key in lookup and lookup[key] != i:
                lookup[key] = None
            else:
                lookup[key] = i
    indices, bad = [], []
    for value in ids:
        value = str(value).replace('\\', '/')
        if value.startswith('./'):
            value = value[2:]
        index = lookup.get(value)
        if index is None:
            bad.append(value)
        else:
            indices.append(index)
    if bad or len(indices) != len(test_paths) or len(set(indices)) != len(test_paths):
        raise ValueError(f'模板 ID 与测试集未一一对应。未匹配示例：{bad[:5]}')
    return np.asarray(indices, dtype=np.int64)

def resolve_submission(test_paths, test_dir, explicit_sample=None, manual_columns=None):
    if len(test_paths) != EXPECTED_TEST_ROWS:
        raise ValueError(f'测试图片应为 {EXPECTED_TEST_ROWS} 张，实际为 {len(test_paths)} 张。')
    if explicit_sample:
        candidates = [Path(explicit_sample)]
    else:
        # 本比赛的模板可能就叫 submission.csv，而不是 sample_submission.csv。
        competition_roots = [p for p in test_dir.parents if p.name == '2026task2']
        candidates = [root / name for root in competition_roots
                      for name in ['sample_submission.csv', 'submission.csv']
                      if (root / name).is_file()]
        if not candidates:
            candidates = [p for p in Path('/kaggle/input').rglob('*')
                          if p.is_file() and p.suffix.lower() == '.csv'
                          and 'sample' in p.stem.lower() and 'sub' in p.stem.lower()]
    matches = []
    for path in sorted(candidates):
        try:
            sample = pd.read_csv(path, dtype=str, keep_default_na=False)
            if list(sample.columns) != ['ID', 'Emotion'] or len(sample) != EXPECTED_TEST_ROWS:
                raise ValueError(f'本任务需要 ID,Emotion 两列和 {EXPECTED_TEST_ROWS} 行')
            id_matches = []
            for column in sample.columns:
                try:
                    order = align_sample_ids(sample[column], test_paths, test_dir)
                    id_matches.append((column, order))
                except ValueError:
                    pass
            if len(id_matches) != 1:
                raise ValueError('无法唯一确定模板中的 ID 列')
            id_col, order = id_matches[0]
            target_col = next(c for c in sample.columns if c != id_col)
            if (id_col, target_col) != ('ID', 'Emotion'):
                raise ValueError('模板 ID 列必须名为 ID，预测列必须名为 Emotion')
            matches.append((sample, id_col, target_col, order, str(path)))
        except (OSError, ValueError, pd.errors.ParserError) as exc:
            if explicit_sample:
                raise ValueError(f'指定的提交模板无效：{path}；{exc}') from exc
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError('找到多个匹配模板，请将 SAMPLE_SUBMISSION 设为具体路径：' +
                         str([m[4] for m in matches]))
    if manual_columns is not None:
        if len(manual_columns) != 2 or len(set(manual_columns)) != 2:
            raise ValueError('MANUAL_SUBMISSION_COLUMNS 应为两个不同的列名，依次为 ID 列和标签列。')
        id_col, target_col = manual_columns
        sample = pd.DataFrame({id_col: [p.name for p in test_paths], target_col: 0})
        return sample, id_col, target_col, np.arange(len(test_paths)), '手动指定，ID 使用完整文件名'
    raise FileNotFoundError('未找到可匹配的 sample_submission.csv。请添加模板并填写路径；'
                            '只有核对列名后才使用 MANUAL_SUBMISSION_COLUMNS。')

def read_gray(path):
    try:
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert('L')
            if image.size != (48, 48):
                raise ValueError(f'尺寸为 {image.size}，与题目 48×48 不符')
            return np.asarray(image, dtype=np.uint8).copy()
    except Exception as exc:
        raise ValueError(f'读取图片失败：{path}；{exc}') from exc

def load_gray_cache(paths, description):
    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 2)) as pool:
        arrays = list(tqdm(pool.map(read_gray, paths), total=len(paths), desc=description))
    return np.stack(arrays)

TRAIN_DIR, TEST_DIR = resolve_data_dirs(TRAIN_DIR, TEST_DIR)
train_paths, y, test_paths, folder_mapping = discover_images(TRAIN_DIR, TEST_DIR)
sample, ID_COL, TARGET_COL, TEST_ORDER, sample_path = resolve_submission(
    test_paths, TEST_DIR, SAMPLE_SUBMISSION, MANUAL_SUBMISSION_COLUMNS)
print('实际数据目录：', TRAIN_DIR.parent)
print('文件夹映射：', folder_mapping)
print('提交模板：', sample_path, '| 列名：', list(sample.columns))
class_counts = pd.DataFrame({'label': range(6), 'class': CLASS_NAMES,
                             'count': np.bincount(y, minlength=6)})
print(class_counts.to_string(index=False))
print(f'训练集 {len(y):,} 张，测试集 {len(test_paths):,} 张')
class_counts.to_csv(OUT / 'class_counts.csv', index=False)
class_counts.plot.bar(x='class', y='count', legend=False, title='Training class distribution')
plt.tight_layout()
plt.savefig(OUT / 'class_distribution.png', dpi=140)
plt.show()
train_images = load_gray_cache(train_paths, '读取训练图片')
test_images = load_gray_cache(test_paths, '读取测试图片')


## 4. 固定验证划分
保持 SEED=42、约12.5%验证集，与旧版使用相同划分流程。完全重复图像按像素哈希分组，避免跨集合泄漏。
没有身份标签，不能保证同一个人的不同照片不跨集合。验证集不参与梯度训练。


In [ ]:
def make_split(labels, hashes, seed=42):
    indices = np.arange(len(labels))
    if len(set(hashes)) == len(hashes):
        tr, va = train_test_split(indices, test_size=0.125, stratify=labels, random_state=seed)
    else:
        splitter = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=seed)
        tr, va = next(splitter.split(indices, labels, groups=hashes))
    if set(labels[tr].tolist()) != set(range(6)) or set(labels[va].tolist()) != set(range(6)):
        raise ValueError('划分后某一类别缺失，需根据实际样本数量调整划分。')
    if set(np.asarray(hashes)[tr]) & set(np.asarray(hashes)[va]):
        raise AssertionError('检测到训练/验证重复泄漏。')
    return np.sort(tr), np.sort(va)

pixel_hashes = [hashlib.sha256(a.tobytes()).hexdigest() for a in train_images]
hash_labels = pd.DataFrame({'hash': pixel_hashes, 'label': y})
print('完全重复的额外图片数：', len(y) - len(set(pixel_hashes)))
print('同一像素对应多个标签的组数：', int((hash_labels.groupby('hash')['label'].nunique() > 1).sum()))
train_idx, val_idx = make_split(y, pixel_hashes, SEED)
print(f'实际训练 {len(train_idx):,} 张；验证 {len(val_idx):,} 张 ({len(val_idx)/len(y):.1%})')
split_table = pd.DataFrame({'file': [p.relative_to(TRAIN_DIR).as_posix() for p in train_paths],
                            'label': y, 'pixel_sha256': pixel_hashes, 'split': 'train'})
split_table.loc[val_idx, 'split'] = 'valid'
print(pd.crosstab(split_table.label, split_table.split).to_string())


RUN_SPEC = dict(version=3, seed=SEED, split=split_table.to_dict(orient='list'))
SIGNATURE = hashlib.sha256(json.dumps(RUN_SPEC, sort_keys=True).encode()).hexdigest()
if (OUT / 'run_config.json').is_file():
    previous = json.loads((OUT / 'run_config.json').read_text(encoding='utf-8'))
    if previous['signature'] != SIGNATURE or previous.get('models') != MODEL_CONFIGS:
        raise ValueError('已有输出使用不同的数据或划分。请修改 RUN_NAME，避免覆盖旧实验。')
split_table.to_csv(OUT / 'split.csv', index=False)
(OUT / 'run_config.json').write_text(json.dumps({
    'version': 3, 'signature': SIGNATURE, 'seed': SEED, 'batch_size': BATCH_SIZE,
    'models': MODEL_CONFIGS, 'total_budget_minutes': TOTAL_MINUTES,
    'prediction_reserve_minutes': PREDICTION_RESERVE_MINUTES,
    'class_mapping': dict(enumerate(CLASS_NAMES)), 'train_size': int(len(train_idx)),
    'validation_size': int(len(val_idx)), 'test_size': int(len(test_paths)),
    'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
}, ensure_ascii=False, indent=2), encoding='utf-8')


## 5. 数据增强
全程保持水平翻转、小幅旋转/平移、亮度对比度变化和少量遮挡。
除分类头热身外，全程使用 MixUp(p=0.60, alpha=0.30)，不再在末期关闭增强。
这是针对原训练准确率升高、验证损失恶化的调整；需要实测确认泛化收益。


In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def transform_for(size, training=False, gentle=False):
    operations = []
    if training:
        operations += [
            T.RandomHorizontalFlip(),
            T.RandomAffine(degrees=8,
                           translate=(0.04, 0.04),
                           scale=(0.95, 1.05),
                           interpolation=InterpolationMode.BILINEAR, fill=128),
            T.ColorJitter(brightness=0.12,
                          contrast=0.12),
        ]
    operations += [T.Resize((size, size), interpolation=InterpolationMode.BILINEAR),
                   T.ToTensor(), T.Normalize(MEAN, STD)]
    if training:
        operations.append(T.RandomErasing(p=0.10,
                                          scale=(0.01, 0.04),
                                          ratio=(0.4, 2.5), value='random'))
    return T.Compose(operations)

class FaceDataset(Dataset):
    def __init__(self, images, indices, transform, labels=None):
        self.images, self.indices = images, np.asarray(indices)
        self.transform, self.labels = transform, labels
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item])
        image = Image.fromarray(self.images[index]).convert('RGB')
        label = -1 if self.labels is None else int(self.labels[index])
        return self.transform(image), label

def seed_worker(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed)
    random.seed(seed)

def make_loader(images, indices, size, training=False, labels=None, seed=SEED, gentle=False):
    kwargs = dict(batch_size=BATCH_SIZE, shuffle=training, num_workers=NUM_WORKERS,
                  pin_memory=DEVICE.type == 'cuda', worker_init_fn=seed_worker,
                  generator=torch.Generator().manual_seed(seed),
                  persistent_workers=NUM_WORKERS > 0, drop_last=False)
    if NUM_WORKERS > 0:
        kwargs['prefetch_factor'] = 2
    return DataLoader(FaceDataset(images, indices, transform_for(size, training, gentle), labels), **kwargs)

y_valid = y[val_idx]

def split_matches(previous_table, current_table):
    columns = ['file', 'label', 'pixel_sha256', 'split']
    if any(column not in previous_table for column in columns):
        return False
    return previous_table[columns].reset_index(drop=True).equals(current_table[columns].reset_index(drop=True))

def valid_probability_array(array, rows):
    return (array.shape == (rows, 6) and np.isfinite(array).all()
            and (array >= 0).all() and np.allclose(array.sum(1), 1, atol=1e-4))

old_candidates = []


## 6. 模型与训练策略
两种标准 ImageNet 预训练 CNN；ResNet34输入160，EfficientNetV2-S输入192，不增加 Transformer。
Dropout=0.4、AdamW权重衰减0.01、标签平滑0.03，使用EMA与混合精度。
第1轮训练分类头，之后训练中后段；ResNet冻结stem与layer1，EfficientNet冻结最前两组features。
第7轮起固定所有BN运行统计量，仍可训练未冻结BN的仿射参数。学习率按余弦下降。
参考：[MixUp原论文](https://arxiv.org/abs/1710.09412)、[PyTorch迁移学习](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial)、
[BatchNorm2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)。这些来源不证明本比赛分数。


In [ ]:
def load_checkpoint(path):
    return torch.load(path, map_location='cpu', weights_only=True)

def atomic_save(payload, path):
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, path)

def cpu_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def build_model(name, pretrained=True):
    if name == 'efficientnet_v2_s':
        builder, weights = models.efficientnet_v2_s, models.EfficientNet_V2_S_Weights.DEFAULT
    elif name == 'resnet34':
        builder, weights = models.resnet34, models.ResNet34_Weights.DEFAULT
    else:
        raise ValueError(name)
    local = LOCAL_PRETRAINED.get(name)
    try:
        model = builder(weights=weights if pretrained and not local else None)
        if pretrained and local:
            model.load_state_dict(load_checkpoint(Path(local)), strict=True)
    except Exception as exc:
        raise RuntimeError(f'{name} 预训练权重加载失败，请启用 Internet 或指定官方 ImageNet 权重路径。') from exc
    if name == 'efficientnet_v2_s':
        model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.classifier[1].in_features, 6))
    else:
        model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.fc.in_features, 6))
    return model

def get_head(model, name):
    return model.classifier if name == 'efficientnet_v2_s' else model.fc

def parallel_model(model):
    return model  # 始终单卡，避免上次 DataParallel 的明显减速

class ModelEMA:
    def __init__(self, model):
        self.model = copy.deepcopy(model).eval()
        self.model.requires_grad_(False)
        self.updates = 0
    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        decay = min(0.99, (1 + self.updates) / (10 + self.updates))
        current = model.state_dict()
        for key, value in self.model.state_dict().items():
            if value.is_floating_point():
                value.lerp_(current[key].detach(), 1 - decay)
            else:
                value.copy_(current[key])

def amp_context():
    return torch.autocast(device_type='cuda', dtype=torch.float16) if DEVICE.type == 'cuda' else nullcontext()

def make_scaler():
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    return torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

@torch.inference_mode()
def predict_views(model, loader, flip=False):
    model.eval()
    original, mirrored = [], []
    for images, _ in loader:
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with amp_context():
            logits = model(images)
        original.append(logits.float().softmax(1).cpu().numpy())
        if flip:
            with amp_context():
                mirrored_logits = model(torch.flip(images, dims=[3]))
            mirrored.append(mirrored_logits.float().softmax(1).cpu().numpy())
    probabilities = np.concatenate(original).astype(np.float32)
    if not valid_probability_array(probabilities, len(loader.dataset)):
        raise ValueError('模型产生了无效预测概率。')
    if flip:
        flipped = np.concatenate(mirrored).astype(np.float32)
        if not valid_probability_array(flipped, len(loader.dataset)):
            raise ValueError('翻转预测概率无效。')
        return probabilities, (probabilities + flipped) / 2
    return probabilities

def metrics_for(probabilities, labels):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    probabilities = probabilities / probabilities.sum(1, keepdims=True)
    return dict(acc=float(accuracy_score(labels, probabilities.argmax(1))),
                macro_f1=float(f1_score(labels, probabilities.argmax(1), average='macro', zero_division=0)),
                loss=float(log_loss(labels, probabilities, labels=list(range(6)))))

def better_score(candidate, incumbent):
    return (candidate['acc'], -candidate['loss']) > (incumbent['acc'], -incumbent['loss'])

def mixup_batch(images, labels, probability, alpha):
    if probability <= 0 or len(labels) < 2 or np.random.random() >= probability:
        return images, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    permutation = torch.randperm(len(labels), device=labels.device)
    return lam * images + (1 - lam) * images[permutation], labels, labels[permutation], lam

def set_train_phase(model, config, epoch):
    head = get_head(model, config['name'])
    model.requires_grad_(True)
    model.train()
    if config['name'] == 'resnet34':
        frozen = [model.conv1, model.bn1, model.layer1]
    else:
        frozen = [model.features[0], model.features[1]]
    for block in frozen:
        block.requires_grad_(False)
        block.eval()
    if epoch < config['warmup_epochs']:
        model.requires_grad_(False)
        head.requires_grad_(True)
        model.eval()
        head.train()
    elif epoch >= config['bn_freeze_after']:
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
    return head


## 7. 开始训练
每个模型最多32轮，最少12轮后才按连续7轮无改进早停；预算不足则提前停止。
raw和ema都是验证准确率。train_aug_acc受MixUp/Dropout影响，不与验证准确率直接作等价比较。
每轮保存best.pt/last.pt/history.csv，每完成一个模型更新备份ZIP。重复运行会复用已完成模型。


In [ ]:
def train_one_model(config, model_index=0):
    name, size = config['name'], config['size']
    model_spec = {key: value for key, value in config.items() if key != 'minutes'}
    folder = OUT / name
    folder.mkdir(exist_ok=True)
    last_path, best_path = folder / 'last.pt', folder / 'best.pt'
    resume = load_checkpoint(last_path) if last_path.is_file() else None
    if resume:
        if resume['signature'] != SIGNATURE or resume['config'] != model_spec:
            raise ValueError(f'{name} 检查点的数据/模型配置不同，请修改 RUN_NAME，不能混用旧实验。')
        if not best_path.is_file():
            raise FileNotFoundError(f'缺少配套最佳权重 {best_path}，请恢复完整输出文件夹。')
        if resume['completed']:
            print(f'{name} 已完成，复用最佳第 {resume["best_epoch"]} 轮。')
            return best_path
    spent_before = float(resume['spent_seconds']) if resume else 0.0
    remaining_phase = max(0.0, config['minutes'] * 60 - spent_before)
    phase_deadline = min(TRAIN_DEADLINE, time.monotonic() + remaining_phase)
    if phase_deadline <= time.monotonic():
        print(name, '训练预算已用完，保留已有最佳权重。')
        return best_path if best_path.is_file() else None
    if model_index > 0 and not resume and phase_deadline - time.monotonic() < 6 * 60:
        print('剩余训练时间不足 6 分钟，跳过辅助模型，保留时间进行预测。')
        return None

    model_seed = SEED + 100 * model_index
    seed_everything(model_seed)
    model = build_model(name, pretrained=resume is None).to(DEVICE, memory_format=torch.channels_last)
    ema = ModelEMA(model)
    runner, ema_runner = parallel_model(model), parallel_model(ema.model)
    head = get_head(model, name)
    head_ids = {id(parameter) for parameter in head.parameters()}
    optimizer = torch.optim.AdamW([
        {'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': config['lr']},
        {'params': list(head.parameters()), 'lr': config['head_lr']},
    ], weight_decay=config['weight_decay'])
    scaler = make_scaler()
    criterion = nn.CrossEntropyLoss(label_smoothing=config['label_smoothing'])
    history, start_epoch = [], 0
    best = dict(acc=-1.0, loss=float('inf'))
    bad_epochs, best_epoch, planned_epochs = 0, 0, config['epochs']
    if resume:
        model.load_state_dict(resume['model'])
        ema.model.load_state_dict(resume['ema'])
        ema.updates = resume['ema_updates']
        optimizer.load_state_dict(resume['optimizer'])
        scaler.load_state_dict(resume['scaler'])
        history, start_epoch = resume['history'], resume['epoch'] + 1
        best, bad_epochs, best_epoch = resume['best'], resume['bad_epochs'], resume['best_epoch']
        planned_epochs = resume['planned_epochs']
        print(f'从 {name} 第 {start_epoch + 1} 轮恢复。')
    del resume
    val_loader = make_loader(train_images, val_idx, size=size, labels=y)
    loader, current_gentle = None, None
    started = time.monotonic()
    print(f'\n{name}: {sum(p.numel() for p in model.parameters())/1e6:.2f}M 参数，输入 {size}，'
          f'最多 {config["epochs"]} 轮，本阶段剩余 {(phase_deadline-started)/60:.1f} 分钟')

    for epoch in range(start_epoch, config['epochs']):
        epoch_started = time.monotonic()
        if epoch_started >= phase_deadline:
            break
        seed_everything(model_seed + epoch)
        gentle = False  # 全程保持增强，不再末期关闭 MixUp
        if loader is None:
            loader = make_loader(train_images, train_idx, size, training=True, labels=y,
                                 seed=model_seed, gentle=False)
        set_train_phase(model, config, epoch)
        if epoch < config['warmup_epochs']:
            lr_factor = 1.0
        else:
            progress = min(1.0, (epoch - config['warmup_epochs']) /
                           max(1, planned_epochs - config['warmup_epochs'] - 1))
            lr_factor = 0.03 + 0.97 * 0.5 * (1 + math.cos(math.pi * progress))
        optimizer.param_groups[0]['lr'] = config['lr'] * lr_factor
        optimizer.param_groups[1]['lr'] = config['head_lr'] * lr_factor
        mix_probability = 0.0 if epoch < config['warmup_epochs'] else config['mixup_p']
        loss_sum = torch.zeros((), device=DEVICE)
        correct_sum = torch.zeros((), device=DEVICE)
        count, time_interrupted = 0, False
        for images, labels in tqdm(loader, desc=f'{name} {epoch+1}/{planned_epochs}', leave=False):
            if time.monotonic() >= phase_deadline:
                time_interrupted = True
                break
            images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
            labels = labels.to(DEVICE, non_blocking=True)
            images, first_labels, second_labels, lam = mixup_batch(
                images, labels, mix_probability, config['mixup_alpha'])
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                logits = runner(images)
                loss = lam * criterion(logits, first_labels) + (1 - lam) * criterion(logits, second_labels)
            if not torch.isfinite(loss):
                raise FloatingPointError('训练 loss 出现 NaN/Inf，请保留日志检查。')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                ema.update(model)
            predictions = logits.detach().argmax(1)
            loss_sum += loss.detach() * len(labels)
            correct_sum += (lam * (predictions == first_labels).sum()
                            + (1 - lam) * (predictions == second_labels).sum())
            count += len(labels)
        if count == 0:
            break

        raw_metrics = metrics_for(predict_views(runner, val_loader), y_valid)
        ema_metrics = metrics_for(predict_views(ema_runner, val_loader), y_valid)
        use_ema = better_score(ema_metrics, raw_metrics)
        validation = ema_metrics if use_ema else raw_metrics
        improved = better_score(validation, best)
        chosen_source = 'ema' if use_ema else 'raw'
        if improved:
            best, bad_epochs, best_epoch = validation.copy(), 0, epoch + 1
            atomic_save(dict(model=cpu_state(ema.model if use_ema else model), signature=SIGNATURE,
                             name=name, epoch=best_epoch, validation=best, image_size=size,
                             class_names=CLASS_NAMES, weight_source=chosen_source), best_path)
        else:
            bad_epochs += 1
        elapsed = time.monotonic() - epoch_started
        total_spent = spent_before + time.monotonic() - started
        history.append(dict(epoch=epoch + 1, train_loss=float(loss_sum.item() / count),
                            train_aug_acc=float(correct_sum.item() / count), val_acc=validation['acc'],
                            val_loss=validation['loss'], val_macro_f1=validation['macro_f1'],
                            raw_acc=raw_metrics['acc'], ema_acc=ema_metrics['acc'],
                            weight_source=chosen_source, gentle=bool(gentle), mixup_p=float(mix_probability),
                            seconds=float(elapsed), backbone_lr=float(optimizer.param_groups[0]['lr']),
                            partial_epoch=bool(time_interrupted), examples_seen=int(count)))
        if epoch >= 1 and not time_interrupted:
            durations = [h['seconds'] for h in history if h['epoch'] > 1 and not h['partial_epoch']]
            expected_seconds = float(np.median(durations))
            remaining = max(0.0, phase_deadline - time.monotonic())
            affordable = int(0.97 * remaining / max(1.0, expected_seconds))
            planned_epochs = min(planned_epochs, max(epoch + 1, epoch + 1 + affordable))
        # 增强策略固定，早停不再依赖一个尚未完成的弱增强阶段。
        early_stop = epoch + 1 >= config['min_epochs'] and bad_epochs >= config['patience']
        time_stop = time_interrupted or time.monotonic() >= phase_deadline or (
            epoch + 1 >= planned_epochs and planned_epochs < config['epochs'])
        completed = early_stop or time_stop or epoch + 1 >= config['epochs']
        reason = 'early_stop' if early_stop else ('time_budget' if time_stop else
                 ('epochs' if completed else 'running'))
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        atomic_save(dict(signature=SIGNATURE, config=model_spec, epoch=epoch, model=cpu_state(model),
                         ema=cpu_state(ema.model), ema_updates=ema.updates, optimizer=optimizer.state_dict(),
                         scaler=scaler.state_dict(), history=history, best=best, best_epoch=best_epoch,
                         bad_epochs=bad_epochs, planned_epochs=planned_epochs, spent_seconds=float(total_spent),
                         completed=bool(completed), reason=reason), last_path)
        print(f'{name} [{epoch+1:02d}/{planned_epochs:02d}] raw={raw_metrics["acc"]:.4f} '
              f'ema={ema_metrics["acc"]:.4f} best={best["acc"]:.4f} '
              f'lr={optimizer.param_groups[0]["lr"]:.2g} MixUp={mix_probability:.2f} '
              f'本轮={elapsed:.0f}s 全程={(time.monotonic()-RUN_STARTED)/60:.1f}min '
              f'{"★" if improved else ""}')
        if completed:
            print('停止原因:', reason, '| 最佳轮数:', best_epoch)
            break
    del runner, ema_runner, model, ema, optimizer, scaler, loader, val_loader
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return best_path if best_path.is_file() else None

trained_models = []
for model_index, config in enumerate(MODEL_CONFIGS):
    best_path = train_one_model(config, model_index)
    if best_path is not None:
        trained_models.append(dict(config=config, best_path=best_path))
        make_backup('已完成 ' + config['name'])
if not trained_models:
    raise RuntimeError('本次没有完成任何新模型，请检查时间预算、GPU 和上方日志。')


## 8. 验证推理方式
比较原图、水平翻转；时间充足时比较160/192双尺寸翻转。测试集使用入选的同一方式。


In [ ]:
def choose_inference(model, config):
    loader = make_loader(train_images, val_idx, config['size'], labels=y)
    plain, flip = predict_views(model, loader, flip=True)
    options = [('original', plain), ('flip', flip)]
    # 多尺度只在还有足够预测余量时尝试；始终用验证集比较。
    remaining = RUN_STARTED + TOTAL_MINUTES * 60 - time.monotonic()
    if MULTISCALE_TTA and config['alt_size'] != config['size'] and remaining > 8 * 60:
        other_loader = make_loader(train_images, val_idx, config['alt_size'], labels=y)
        _, other_flip = predict_views(model, other_loader, flip=True)
        options.append(('multiscale_flip', (flip + other_flip) / 2))
        del other_loader
    scores = [dict(mode=mode, **metrics_for(probability, y_valid)) for mode, probability in options]
    best_index = max(range(len(options)), key=lambda i: (scores[i]['acc'], -scores[i]['loss']))
    mode, probability = options[best_index]
    del loader
    return mode, probability, scores

def test_probabilities(model, config, mode):
    loader = make_loader(test_images, np.arange(len(test_images)), config['size'])
    if mode == 'original':
        probabilities = predict_views(model, loader)
    else:
        _, probabilities = predict_views(model, loader, flip=True)
    if mode == 'multiscale_flip':
        other_loader = make_loader(test_images, np.arange(len(test_images)), config['alt_size'])
        _, other = predict_views(model, other_loader, flip=True)
        probabilities = (probabilities + other) / 2
        del other_loader
    del loader
    return probabilities

model_runs = []
inference_rows = []
for item in trained_models:
    config, best_path = item['config'], item['best_path']
    checkpoint = load_checkpoint(best_path)
    if checkpoint['signature'] != SIGNATURE or checkpoint['class_names'] != CLASS_NAMES:
        raise ValueError('检查点与当前数据或标签映射不同。')
    model = build_model(config['name'], pretrained=False).to(DEVICE, memory_format=torch.channels_last)
    model.load_state_dict(checkpoint['model'])
    runner = parallel_model(model)
    mode, valid_probability, rows = choose_inference(runner, config)
    inference_rows.extend(dict(model=config['name'], **row) for row in rows)
    model_runs.append(dict(name=config['name'], valid=valid_probability, test=None,
                           source='v3', mode=mode, config=config, checkpoint=str(best_path),
                           best_epoch=int(checkpoint['epoch']), weight_source=checkpoint['weight_source']))
    print(config['name'], '| 推理方式:', mode, '| 验证:', metrics_for(valid_probability, y_valid))
    del model, runner, checkpoint
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

new_model_count = len(model_runs)
for previous in old_candidates:
    inference_rows.append(dict(model=previous['name'], mode=previous['mode'],
                               **metrics_for(previous['valid'], y_valid)))
model_runs.extend(old_candidates)
inference_table = pd.DataFrame(inference_rows)
inference_table.to_csv(OUT / 'inference_comparison.csv', index=False)
print('\n各模型验证结果（还不是 Kaggle 分数）：')
print(inference_table.to_string(index=False))


## 9. 生成提交
比较单模型及25%/50%/75%两模型概率融合。验证集参与了轮次、推理方式和权重选择，存在选型偏差。
生成submission_v3.csv及备用单模型submission_v3_single.csv；两份相同则不必重复提交。
历史0.72结果只作对照，不把它的预测标签当训练真值。


In [ ]:
def save_submission(template, id_column, target_column, probabilities, order, destination):
    if (id_column, target_column) != ('ID', 'Emotion'):
        raise ValueError('提交列名必须为 ID,Emotion。')
    if len(template) != EXPECTED_TEST_ROWS:
        raise ValueError(f'提交应为 {EXPECTED_TEST_ROWS} 行，实际模板只有 {len(template)} 行。')
    actual_paths = sorted(p for p in TEST_DIR.rglob('*')
                          if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if (len(actual_paths) != EXPECTED_TEST_ROWS or len(test_paths) != EXPECTED_TEST_ROWS
            or {p.resolve() for p in actual_paths} != {p.resolve() for p in test_paths}):
        raise ValueError('当前预测使用的图片不是完整的 7065 张测试图片，请重新读取测试集并推理。')
    expected_order = align_sample_ids(template[id_column], test_paths, TEST_DIR)
    if not np.array_equal(order, expected_order):
        raise ValueError('提交的 ID 顺序与预测使用的图片顺序不一致。')
    if probabilities.shape != (len(template), 6) or not np.isfinite(probabilities).all():
        raise ValueError('预测概率的维度或数值不正确。')
    if (probabilities < 0).any() or not np.allclose(probabilities.sum(1), 1, atol=1e-4):
        raise ValueError('每行必须是有效的六分类概率。')
    if len(order) != len(template) or not np.array_equal(np.sort(order), np.arange(len(template))):
        raise ValueError('模板与预测行未一一对应。')
    output = template.copy()
    output[target_column] = probabilities[np.asarray(order)].argmax(1).astype(np.int64)
    if output[id_column].duplicated().any() or not output[target_column].between(0, 5).all():
        raise ValueError('提交中存在重复 ID 或越界标签。')
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    output.to_csv(destination, index=False)
    checked = pd.read_csv(destination, dtype={id_column: str}, keep_default_na=False)
    assert list(checked.columns) == list(template.columns)
    assert checked[id_column].tolist() == template[id_column].astype(str).tolist()
    assert checked[target_column].tolist() == output[target_column].tolist()
    return output

def select_blend(runs, labels):
    count = len(runs)
    if count == 0:
        raise ValueError('没有候选模型。')
    candidates = []
    for index in range(count):
        weights = [0.0] * count
        weights[index] = 1.0
        candidates.append(weights)
    # 最终最多融合两个模型，避免推理成本和验证集调参规模不断增加。
    for first in range(count):
        for second in range(first + 1, count):
            for proportion in [0.25, 0.5, 0.75]:
                weights = [0.0] * count
                weights[first], weights[second] = proportion, 1 - proportion
                candidates.append(weights)
    rows, selected = [], None
    for weights in candidates:
        probability = sum(w * run['valid'] for w, run in zip(weights, runs) if w > 0)
        score = metrics_for(probability, labels)
        result = dict(weights=weights, **score)
        rows.append(result)
        if selected is None or better_score(result, selected):
            selected = result
    return selected, rows

selected, blend_rows = select_blend(model_runs, y_valid)
blend_table = pd.DataFrame(blend_rows)
blend_table.to_csv(OUT / 'blend_comparison.csv', index=False)
best_new_index = max(range(new_model_count), key=lambda index: (
    metrics_for(model_runs[index]['valid'], y_valid)['acc'],
    -metrics_for(model_runs[index]['valid'], y_valid)['loss']))
required_indices = {i for i, weight in enumerate(selected['weights']) if weight > 0} | {best_new_index}
for index in sorted(required_indices):
    run = model_runs[index]
    if run['test'] is not None:
        continue
    checkpoint = load_checkpoint(Path(run['checkpoint']))
    model = build_model(run['name'], pretrained=False).to(DEVICE, memory_format=torch.channels_last)
    model.load_state_dict(checkpoint['model'])
    runner = parallel_model(model)
    run['test'] = test_probabilities(runner, run['config'], run['mode'])
    np.savez_compressed(OUT / run['name'] / 'predictions.npz', validation_indices=val_idx,
                        labels=y_valid, valid_selected=run['valid'], test_selected=run['test'],
                        test_ids=np.asarray([p.name for p in test_paths]))
    del model, runner, checkpoint
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

final_valid = sum(w * run['valid'] for w, run in zip(selected['weights'], model_runs) if w > 0)
final_test = sum(w * run['test'] for w, run in zip(selected['weights'], model_runs) if w > 0)
submission_path = Path('/kaggle/working/submission_v3.csv')
submission = save_submission(sample, ID_COL, TARGET_COL, final_test, TEST_ORDER, submission_path)
shutil.copy2(submission_path, OUT / 'submission_v3.csv')
single_path = Path('/kaggle/working/submission_v3_single.csv')
single_submission = save_submission(sample, ID_COL, TARGET_COL, model_runs[best_new_index]['test'],
                                     TEST_ORDER, single_path)
shutil.copy2(single_path, OUT / 'submission_v3_single.csv')
selection = dict(
    models=[dict(name=r['name'], source=r['source'], inference=r['mode'],
                 best_epoch=r.get('best_epoch'), weight_source=r.get('weight_source')) for r in model_runs],
    **selected, best_new_single=model_runs[best_new_index]['name'],
    submissions_identical=bool(submission.equals(single_submission)),
    elapsed_minutes=float((time.monotonic() - RUN_STARTED) / 60),
    validation_size=int(len(val_idx)), test_size=int(len(test_paths)),
    old_reported_kaggle_score=0.72, new_kaggle_score=None,
    note='验证集参与了模型及融合选择；新版 Kaggle 得分仍需实际提交确认。',
)
(OUT / 'selection.json').write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8')
print('\n模型顺序：', [r['name'] for r in model_runs])
print('所选权重：', selected['weights'])
print('选型验证准确率：', round(selected['acc'], 4), '（不是 Kaggle 成绩）')
print('首选提交：', submission_path)
print('备用新单模型：', single_path, '| 模型：', selection['best_new_single'])
print('两份提交是否相同：', selection['submissions_identical'])
print('实际行数：', len(submission), '| 列名：', list(submission.columns))
print(submission.head().to_string(index=False))
print(f'当前耗时 {selection["elapsed_minutes"]:.1f} 分钟；没有覆盖旧版 submission.csv。')

print('历史最佳 Kaggle：0.72；本次 Kaggle 分数：尚未提交。')
print('本次选型验证值不能直接当成 Kaggle 成绩，也不能保证超过历史提交。')
make_backup('提交文件与预测缓存已生成')


## 10. 报告与最终下载
输出真实运行日志、每类指标、混淆矩阵和曲线。务必下载提交CSV与备份ZIP。


In [ ]:
report = classification_report(y_valid, final_valid.argmax(1), labels=list(range(6)),
                               target_names=CLASS_NAMES, digits=4, zero_division=0)
print(report)
(OUT / 'classification_report.txt').write_text(report, encoding='utf-8')
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_valid, final_valid.argmax(1), labels=list(range(6)),
                                        display_labels=CLASS_NAMES, cmap='Blues', ax=ax,
                                        colorbar=False, xticks_rotation=35)
ax.set_title('V3 validation confusion matrix')
fig.tight_layout()
fig.savefig(OUT / 'confusion_matrix.png', dpi=140)
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for item in trained_models:
    name = item['config']['name']
    history = pd.read_csv(OUT / name / 'history.csv')
    axes[0].plot(history.epoch, history.raw_acc, '--', label=name + ' raw')
    axes[0].plot(history.epoch, history.ema_acc, label=name + ' EMA')
    axes[1].plot(history.epoch, history.val_loss, label=name)
axes[0].set(xlabel='Epoch', ylabel='Validation accuracy', title='Raw vs EMA')
axes[1].set(xlabel='Epoch', ylabel='Cross entropy', title='Validation loss')
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUT / 'training_curves.png', dpi=140)
plt.show()

summary = f'''# 人脸表情识别：v3 运行结果

目标：在两小时以内的运行预算下，争取超过旧版 Kaggle 0.72，并以 0.75 为下一目标。
目标不是已达到的成绩；新版 Kaggle 分数：待实际提交后填写。

样本：训练数据 {len(y)} 张，划分后用于梯度更新 {len(train_idx)} 张，验证 {len(val_idx)} 张，测试 {len(test_paths)} 张。
图像：48×48 灰度复制成三通道，ResNet34输入160，EfficientNetV2-S输入192，ImageNet 标准化。
模型：ImageNet 预训练 EfficientNetV2-S 和 ResNet34；以实际完成的模型为准。
策略：第一轮训练分类头，后续微调中后段、冻结早期层，AdamW、余弦学习率、0.03 标签平滑、MixUp(p=0.60, alpha=0.30)、EMA。
后期保持增强及 MixUp；Dropout=0.4、weight_decay=0.01；第7轮起固定 BN 统计量。每轮比较普通权重和 EMA。
测试时以验证结果选择原图、水平翻转或两种尺寸的平均概率；最多融合两个模型。
旧模型权重已丢失，本版独立训练，不读取旧会话变量，也不把旧 CSV 标签当作真值。

模型顺序：{[r['name'] for r in model_runs]}。
融合权重：{selected['weights']}。
选型验证准确率：{selected['acc']:.4f}；Macro-F1：{selected['macro_f1']:.4f}。
该验证集参与选型，不是独立测试评估。没有使用测试标签，也没有将验证图片加入梯度训练。
像素哈希只能排除完全重复图像跨集合，无法保证同一个人的不同照片不跨集合。
至生成 CSV 的耗时：{selection['elapsed_minutes']:.1f} 分钟。

首选文件：submission_v3.csv；备用新单模型：submission_v3_single.csv。
两份文件相同：{selection['submissions_identical']}。
已检查列名 ID/Emotion、预测范围 0～5、文件名顺序与图片数量。
'''
(OUT / 'experiment_summary.md').write_text(summary, encoding='utf-8')
print('请保存运行版本及全部输出。反馈结果时，提供 inference_comparison.csv、selection.json 和实际 Kaggle 分数。')

make_backup('最终完整备份', verify=True)
from IPython.display import display, FileLink
print('必须下载下面两个文件到电脑；压缩包只存在临时磁盘时仍可能丢失。')
display(FileLink('/kaggle/working/submission_v3.csv'))
display(FileLink(str(BACKUP_PATH)))
print('备份大小：', round(BACKUP_PATH.stat().st_size / 1024**2), 'MiB')


## 下次恢复，无需旧内存变量
1. 保留电脑上的本 Notebook 和 `fer6_v3_backup.zip`。
2. 将备份作为输入数据添加到新的 Kaggle Notebook，同时添加原比赛数据。
3. 若输入目录只有一个同名ZIP或解压后的fer6_score_v3目录，本程序自动找到它；否则把完整路径填到第1节的 `RESUME_FROM`。
4. 从头运行所有单元。程序重建变量和图像缓存，核对数据与配置；完成的模型跳过训练，中断的模型恢复已保存轮次。
5. 恢复版本仍需使用相同模型配置。保存版本与下载备份属于输出保存操作，不等于只保存代码文本。

若中途手动停止，last.pt记录的是最后一个完整保存的轮次，当前未保存批次可能需重跑。
自动ZIP在模型完成时更新，训练中途的最新last.pt可能晚于ZIP。重新运行训练单元可继续；不要删除本地输出。

Kaggle官方说明：[Notebooks](https://www.kaggle.com/docs/notebooks)。
